> **Local runner:** Requires the public UCI CSV and matching reconstruction settings. Executable scripts are in `scripts/`; saved aggregate references are in `evidence/reconstruction/`.

# Characterize the reconstructed weight-dependent partitions

The selected fits must match the saved reconstruction evidence before the results are used in the manuscript.

In [ ]:
from pathlib import Path

def repository_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "scripts" / "validate_experiment_evidence.py").is_file():
            return folder
    raise FileNotFoundError("Open Jupyter from inside the repository directory")

ROOT = repository_root()
import subprocess, sys
from IPython.display import display
import pandas as pd
RAW_CSV = Path(r"C:\Projects\data\diabetic_data.csv")  # Set this path
RECON = ROOT / "evidence" / "reconstruction"
SCRIPT = ROOT / "scripts" / "characterize_weighted_clusters.py"
VALIDATOR = ROOT / "scripts" / "analyze_weight_scaling.py"
OUT = ROOT / "outputs" / "weight_characterization"
assert RAW_CSV.is_file(), f"Public UCI file not found: {RAW_CSV}"
assert SCRIPT.is_file() and VALIDATOR.is_file(), "Research scripts are missing"

In [ ]:
command = [
    sys.executable, str(SCRIPT),
    "--raw-csv", str(RAW_CSV),
    "--validator", str(VALIDATOR),
    "--reference-fit-csv", str(RECON / "fit_level_seed_runs_ninit20_REFERENCE.csv"),
    "--reference-cross-csv", str(RECON / "cross_weight_partition_agreement_ninit20_REFERENCE.csv"),
    "--out", str(OUT),
    "--n-init", "20", "--seeds-per-weight", "40",
    "--scaler", "robust", "--weight-transform", "direct", "--seed", "42",
]
# Add --split-manifest if the saved reconstruction used a specific split.
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Characterization did not complete")

In [ ]:
out=OUT
for f in ["feature_block_concentration.csv","cluster_age_utilization_summary.csv","diagnosis_age_cluster_contrasts.csv","aligned_transition.csv"]:
 print("\n===",f,"===")
 d=pd.read_csv(out/f)
 display(d.head(30) if "contrasts" in f else d)
print("Check characterization_manifest.json: any mismatch with saved 06C reconstruction makes this result UNSUITABLE for the manuscript.")